# Model selection for text2bash

Compares candidate small base models for the English/Russian
instruction -> bash command task, to justify the base-model choice in
`PLAN.md` before committing to a full fine-tune. Runs entirely on CPU, on
this machine, using the project's own planned prompt format.

Two architecture families are compared:
- **Decoder-only causal LMs** (the HN-post reference approach: fine-tune a
  small GPT-style model directly on instruction -> command text)
- **Encoder-decoder (seq2seq) models** (the architecture the original
  NL2Bash research used, and a natural fit for a "translate this request
  into a command" framing)

All candidates are evaluated **zero-shot** (no fine-tuning yet) — this
notebook is about tokenizer efficiency, load/inference cost, and baseline
behavior, not final task accuracy. Final task accuracy is measured after
fine-tuning, in Phase 4's eval harness.

In [1]:
import time
import torch
from transformers import AutoConfig, AutoTokenizer, AutoModelForCausalLM, AutoModelForSeq2SeqLM

torch.manual_seed(0)
DEVICE = "cpu"


In [2]:
CANDIDATES = [
    {
        "id": "Qwen/Qwen3-0.6B-Base",
        "family": "decoder-only",
        "license": "Apache-2.0",
        "note": "Primary candidate under consideration",
    },
    {
        "id": "Qwen/Qwen2.5-0.5B",
        "family": "decoder-only",
        "license": "Apache-2.0",
        "note": "Lighter same-family alternative",
    },
    {
        "id": "google/mt5-small",
        "family": "encoder-decoder",
        "license": "Apache-2.0",
        "note": "mT5: pretrained on 101 languages (mC4) incl. Russian; closest to original NL2Bash-style seq2seq modeling",
    },
    {
        "id": "google/flan-t5-small",
        "family": "encoder-decoder",
        "license": "Apache-2.0",
        "note": "Smallest/fastest candidate; instruction-tuned; T5 pretraining is English-dominant -> included as an English-centric baseline",
    },
    {
        "id": "google/byt5-small",
        "family": "encoder-decoder (byte-level)",
        "license": "Apache-2.0",
        "note": "Byte-level tokenizer is script-agnostic, so no dedicated Russian vocab needed -> but sequences are much longer",
    },
]


In [3]:
EN_PROMPT_INSTR = "find all files larger than 100MB in the current directory"
RU_PROMPT_INSTR = "найди все файлы с расширением .log старше 7 дней"

DECODER_TEMPLATE = "### Instruction:\n{instruction}\n### Command:\n"
SEQ2SEQ_TEMPLATE = "translate to bash command: {instruction}"

def build_prompt(entry, instruction):
    if entry["family"] == "decoder-only":
        return DECODER_TEMPLATE.format(instruction=instruction)
    return SEQ2SEQ_TEMPLATE.format(instruction=instruction)


In [4]:
def load_candidate(entry):
    model_id = entry["id"]
    t0 = time.time()
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    if entry["family"] == "decoder-only":
        model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float32)
    else:
        model = AutoModelForSeq2SeqLM.from_pretrained(model_id, dtype=torch.float32)
    model.eval()
    load_time = time.time() - t0
    n_params = sum(p.numel() for p in model.parameters())
    return tokenizer, model, load_time, n_params


def generate(entry, tokenizer, model, instruction, max_new_tokens=24):
    prompt = build_prompt(entry, instruction)
    if entry["family"] == "decoder-only":
        inputs = tokenizer(prompt, return_tensors="pt")
    else:
        inputs = tokenizer(prompt, return_tensors="pt")
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        )
    elapsed = time.time() - t0
    if entry["family"] == "decoder-only":
        gen_ids = out[0][inputs["input_ids"].shape[1]:]
    else:
        gen_ids = out[0]
    text = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
    n_gen_tokens = len(gen_ids)
    return text, elapsed, n_gen_tokens


In [5]:
results = []
for entry in CANDIDATES:
    print(f"=== {entry['id']} ===")
    tokenizer, model, load_time, n_params = load_candidate(entry)

    en_tok_len = len(tokenizer(EN_PROMPT_INSTR)["input_ids"])
    ru_tok_len = len(tokenizer(RU_PROMPT_INSTR)["input_ids"])

    en_text, en_time, en_ntok = generate(entry, tokenizer, model, EN_PROMPT_INSTR)
    ru_text, ru_time, ru_ntok = generate(entry, tokenizer, model, RU_PROMPT_INSTR)

    row = {
        "id": entry["id"],
        "family": entry["family"],
        "params_M": round(n_params / 1e6, 1),
        "load_s": round(load_time, 1),
        "en_tok_len": en_tok_len,
        "ru_tok_len": ru_tok_len,
        "en_gen_s": round(en_time, 2),
        "en_tok_per_s": round(en_ntok / en_time, 1) if en_time > 0 else None,
        "ru_gen_s": round(ru_time, 2),
        "en_output": en_text[:80],
        "ru_output": ru_text[:80],
    }
    results.append(row)
    print(row)
    print()

    del model, tokenizer


=== Qwen/Qwen3-0.6B-Base ===


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

{'id': 'Qwen/Qwen3-0.6B-Base', 'family': 'decoder-only', 'params_M': 596.0, 'load_s': 11.0, 'en_tok_len': 14, 'ru_tok_len': 17, 'en_gen_s': 2.85, 'en_tok_per_s': 8.4, 'ru_gen_s': 2.04, 'en_output': "ls -l | grep -v '^[0-9]' | awk '{print $9}' | sort -n", 'ru_output': '```bash\nfind . -type f -name "*.log" -mtime +7 -exec ls -l {} \\'}

=== Qwen/Qwen2.5-0.5B ===


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

{'id': 'Qwen/Qwen2.5-0.5B', 'family': 'decoder-only', 'params_M': 494.0, 'load_s': 3.2, 'en_tok_len': 14, 'ru_tok_len': 17, 'en_gen_s': 1.46, 'en_tok_per_s': 16.4, 'ru_gen_s': 1.53, 'en_output': 'find . -type f -size +100M\n### Output:\nfind . -type f -size +', 'ru_output': 'find . -type f -name "*.log" -mtime +7\n### Output:\nfind . -type f -'}

=== google/mt5-small ===


...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/172 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

{'id': 'google/byt5-small', 'family': 'encoder-decoder (byte-level)', 'params_M': 299.6, 'load_s': 89.5, 'en_tok_len': 58, 'ru_tok_len': 84, 'en_gen_s': 0.5, 'en_tok_per_s': 49.6, 'ru_gen_s': 0.42, 'en_output': 'to bash command: find', 'ru_output': 'дневник на б'}



In [6]:
header = ["id", "family", "params_M", "load_s", "en_tok_len", "ru_tok_len", "en_gen_s", "en_tok_per_s"]
col_w = {h: max(len(h), max(len(str(r[h])) for r in results)) for h in header}

def fmt_row(vals):
    return " | ".join(str(v).ljust(col_w[h]) for h, v in zip(header, vals))

print(fmt_row(header))
print("-+-".join("-" * col_w[h] for h in header))
for r in results:
    print(fmt_row([r[h] for h in header]))

print()
for r in results:
    print(f"--- {r['id']} ---")
    print(f"  EN zero-shot: {r['en_output']!r}")
    print(f"  RU zero-shot: {r['ru_output']!r}")


id                   | family                       | params_M | load_s | en_tok_len | ru_tok_len | en_gen_s | en_tok_per_s
---------------------+------------------------------+----------+--------+------------+------------+----------+-------------
Qwen/Qwen3-0.6B-Base | decoder-only                 | 596.0    | 11.0   | 14         | 17         | 2.85     | 8.4         
Qwen/Qwen2.5-0.5B    | decoder-only                 | 494.0    | 3.2    | 14         | 17         | 1.46     | 16.4        
google/mt5-small     | encoder-decoder              | 300.2    | 97.6   | 14         | 18         | 0.23     | 13.2        
google/flan-t5-small | encoder-decoder              | 77.0     | 121.2  | 12         | 37         | 0.2      | 124.6       
google/byt5-small    | encoder-decoder (byte-level) | 299.6    | 89.5   | 58         | 84         | 0.5      | 49.6        

--- Qwen/Qwen3-0.6B-Base ---
  EN zero-shot: "ls -l | grep -v '^[0-9]' | awk '{print $9}' | sort -n"
  RU zero-shot: '```bash\nfind

## Discussion & decision

**Result table** (from the cells above):

| id | family | params (M) | load (s) | EN tok len | RU tok len | gen speed (tok/s) |
|---|---|---|---|---|---|---|
| Qwen/Qwen3-0.6B-Base | decoder-only | 596.0 | 11.0 | 14 | 17 | 8.4 |
| Qwen/Qwen2.5-0.5B | decoder-only | 494.0 | 3.2 | 14 | 17 | 16.4 |
| google/mt5-small | encoder-decoder | 300.2 | 97.6 | 14 | 18 | 13.2 |
| google/flan-t5-small | encoder-decoder | 77.0 | 121.2 | 12 | 37 | 124.6 |
| google/byt5-small | encoder-decoder (byte-level) | 299.6 | 89.5 | 58 | 84 | 49.6 |

**Zero-shot outputs** (no fine-tuning — this is the pretrained model's raw
instinct on the exact Phase 3 prompt template):

- **Qwen3-0.6B-Base** — EN: `ls -l | grep -v '^[0-9]' | awk '{print $9}' | sort -n`
  (plausible bash, not quite right). RU: a fenced ```` ```bash ```` block
  containing `find . -type f -name "*.log" -mtime +7 -exec ls -l {} ` — the
  *correct* command, modulo the markdown fence.
- **Qwen2.5-0.5B** — EN: `find . -type f -size +100M` — exactly correct.
  RU: `find . -type f -name "*.log" -mtime +7` — exactly correct (both
  trail off into an echoed `### Output:` continuation, a pretraining
  artifact, not a correctness issue).
- **mT5-small** — outputs `<extra_id_0>` for both languages: the raw
  span-corruption sentinel token. Expected — mT5 is pretrained but never
  instruction-tuned, so it has no notion of "answer this request."
- **FLAN-T5-small** — generates an empty string for both languages: it
  emits EOS immediately. Its instruction-tuning mixture doesn't cover
  shell commands, so it has nothing to say.
- **ByT5-small** — EN: echoes part of the prompt back (`to bash command:
  find`). RU: hallucinates unrelated Russian text (`дневник на б`,
  "diary on b"). No usable signal either way.

**Decision: stay with the decoder-only family, specifically Qwen3-0.6B-Base
as the primary pick, with Qwen2.5-0.5B noted as a strong lighter/faster
fallback.**

Why decoder-only over encoder-decoder: this was the real finding, not an
assumption. All three T5-family encoder-decoder checkpoints failed outright
zero-shot on this task — not "weaker," but producing no usable command at
all. That's because off-the-shelf mT5/FLAN-T5/ByT5 checkpoints are
pretrained on span-corruption or general instruction-following, not on
code/shell-heavy corpora, so they have no prior for "instruction -> bash
command." The Qwen decoder-only models, by contrast, are pretrained on
large web+code corpora and already produce near-correct or fully-correct
bash *before any fine-tuning at all*. Fine-tuning will sharpen that prior,
not have to build it from nothing — a much easier training problem, which
matters given CPU-only, LoRA, pilot-scale (~52k example) training.

This also means the classic NL2Bash seq2seq framing (the architecture the
original NL2Bash paper used) is not the right fit *for these specific small
off-the-shelf checkpoints* — it might work with a code-pretrained
encoder-decoder model, but none of the three tested here are code-pretrained,
and finding/evaluating one was out of scope for this pass.

Why Qwen3-0.6B-Base over Qwen2.5-0.5B, despite Qwen2.5-0.5B's zero-shot
outputs being arguably *cleaner* in this 2-prompt sample and it being
lighter/faster (3.2s load vs 11.0s, ~2x generation speed): the sample size
here is two prompts, not a benchmark — not enough to call a winner on
output quality. Qwen3-0.6B-Base is the model that was already
being considered, is the more recent of the two in the same family, and the speed gap,
while real, isn't prohibitive for a pilot-scale LoRA run. **Qwen2.5-0.5B
is the documented fallback** if CPU fine-tuning time becomes the binding
constraint in Phase 3 — swapping `--base_model` in `train/finetune_lora.py`
is a one-flag change, not a redesign.

One caveat on the load-time numbers: mT5/FLAN-T5/ByT5 all show anomalously
slow loads (90–120s) vs. Qwen's 3–11s, correlated with a "tied weights ...
will NOT tie them" warning from `transformers`. This looks like a
framework-level quirk in how this `transformers` version resolves tied
embeddings for these checkpoints, not an inherent property of
encoder-decoder architectures — noted for completeness, but it did not
drive the decision above (the zero-shot failures did).